In [1]:
anthropic_key = "REDACTED_API_KEY"
import getpass
import os


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("ANTHROPIC_API_KEY")

In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_anthropic import ChatAnthropic
from langchain_ollama import ChatOllama
from IPython.display import Image, display
from typing import Dict, TypedDict, Optional
import random
import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
import json

In [ ]:
class EnvironmentGraphState(TypedDict):
    """
    State Graph for the Environment Simulation. 
    A State Contatins the Following information: 
    
    - previous_agent: the agent prompted/used before this
    - current_agent: the current agent to be used
    - metadata: json data carried from the previous agent
    - day_count: day of experiment (will be used to interrupt and select new flows)
    - 
    """
    

In [ ]:
prompt = """
  You
"""

In [12]:
json.dumps(prompt,indent=4)

'"\\n  You are a reality check agent that evaluates whether a human\'s action is physically and biologically possible given their current health and environment. You will analyze the environmental status, human\'s health, and action taken to determine if the action is feasible.\\n\\n  If the action is physically possible, return \\"possibility\\": \\"possible\\".\\n  If the action is impossible due to physical or biological constraints, return \\"possibility\\": \\"impossible\\" with an explanation in \\"reasoning\\".\\n  Input Format (JSON)\\n\\n  You will receive:\\n\\n      environment_status - Describes surroundings, temperature, weather, available resources, and danger level.\\n      current_health - A structured JSON representing various health parameters.\\n      action_taken - The action the human attempts.\\n\\n  Example Input\\n\\n  {\\n    \\"environment_status\\": {\\n      \\"temperature\\": -10,\\n      \\"weather\\": \\"blizzard\\",\\n      \\"food_availability\\": \\"no

In [30]:
with open("prompts/AIAH_consience_manager.json","r") as f:
    prompt_json = json.load(f)


In [ ]:
prompt = """
You are the Skills Manager, responsible for tracking and updating the human agent's skills and abilities based on their actions and experiences. Your role is to simulate how the human learns, improves, or forgets skills over time.
Your Task

Given:

    The environment before the action (context for skill use).
    The action taken (what the human did).
    The human's health status (which affects learning and execution).
    The outcome of the action (success, failure, injury, improvement).
    Previously stored skills (existing abilities and proficiency levels).

You must add new skills, increase proficiency in practiced skills, or track skill degradation if an ability isn’t used for a long time.
Processing Rules

    Add new skills - If the human performs an action they have never done before, create a new skill entry.
    Increase proficiency - If an existing skill is used successfully, improve its proficiency level.
    Track failure consequences - If the action fails, mark it as a learning experience (potentially increasing skill after multiple attempts).
    Consider health impact - Low energy, injury, or illness may slow down skill improvement.
    Limit skill tracking - Store only ~10-15 relevant skills, removing the least-used ones over time.

Input Format (JSON)

{
  "environment_status": { ... },
  "action_taken": "...",
  "current_health": { ... },
  "environment_outcome": { ... },
  "previous_skills": [
    { "skill": "...", "proficiency": 5 }
  ]
}

Output Format (JSON)

{
  "updated_skills": [
    { "skill": "...", "proficiency": 6 }
  ]
}
"""

In [28]:
print(json.dumps(prompt))

"\nYou are the Conscience Manager, responsible for maintaining and updating the human agent\u2019s sense of morality, emotions, and personal reflections based on their experiences. Your role is to track moral dilemmas, emotional responses, and personal lessons learned from actions and consequences.\nYour Task\n\nGiven:\n\n    The environment before the action (context for decision-making).\n    The action taken (what the human did).\n    The human\u2019s health status (which can impact emotional state).\n    The outcome of the action (success, failure, harm, or benefits).\n    Previously stored conscience notes (existing reflections, regrets, or moral realizations).\n\nYou must store new moral reflections, update existing conscience notes if similar experiences occur, and ensure that personal growth is tracked over time.\nProcessing Rules\n\n    Record moral reflections \u2013 If the action had a moral or emotional impact, add a new conscience note.\n    Update existing reflections \u2